# Reinforcement Learning Chapter 2: Multi-Armed Bandits
This notebook reproduces the parameter study of various multi-armed bandit algorithms (Figure 2.6) from Chapter 2 of the textbook *Reinforcement Learning: An Introduction* by Sutton & Barto (2nd Edition).

We evaluate and compare the following algorithms on a 10-armed testbed:
1. **$\epsilon$-greedy Action Selection**
2. **Greedy Action Selection with Optimistic Initial Values**
3. **Upper Confidence Bound (UCB) Action Selection**
4. **Gradient Bandit Algorithms**


## 1. $\epsilon$-Greedy Action Selection
The $\epsilon$-greedy method balances exploration and exploitation. With probability $1 - \epsilon$, the agent selects the greedy action (the action with the highest estimated value $Q_t(a)$). With probability $\epsilon$, the agent selects an action uniformly at random.

The action value estimates $Q_t(a)$ are updated using sample averages:
$$Q_{t+1}(A_t) = Q_t(A_t) + \frac{1}{N_t(A_t)} [R_t - Q_t(A_t)]$$
where $N_t(A_t)$ is the number of times action $A_t$ has been selected.


In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

def eGreedy(k=10, iterations=1000, epsilon=0.2, reward_std=1.0, seed=None):
    # Set seed for reproducibility
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)
    
    # Initialize true action values
    initial_action_values = np.array([random.gauss(0, 1) for _ in range(k)])
    
    # Initialize estimates and counts
    Q = np.zeros(k)
    N = np.zeros(k)
    
    rewards = []
    average_reward = []
    opt_count = 0
    opt = []

    best_action = np.argmax(initial_action_values)
    
    for i in range(iterations):
        # Epsilon-greedy action selection
        if random.uniform(0, 1) < epsilon:
            action = random.randint(0, k - 1)
        else:
            # Random tie-breaking
            max_Q = np.max(Q)
            action = np.random.choice(np.flatnonzero(Q == max_Q))
        
        # Track optimal action selection
        if action == best_action:
            opt_count += 1
        opt.append(opt_count / (i + 1))
        
        # Generate reward (correct Gaussian reward without clipping)
        reward = np.random.normal(initial_action_values[action], reward_std)
        
        # Update action-value estimate (sample-average)
        N[action] += 1
        Q[action] += (reward - Q[action]) / N[action]
        
        rewards.append(reward)
        average_reward.append(np.mean(rewards))
    
    return rewards, average_reward, opt

# Parameters
iterations = 1000
k = 10

random.seed(1223)

# --- Plotting section ---
epsilons = [0, 0.01, 0.1, 0.2, 0.5]
iterations = 1000

plt.figure(figsize=(10, 6))

for eps in epsilons:
    _, avg_reward, _ = eGreedy(k=10, iterations=iterations, epsilon=eps, seed=423)
    plt.plot(avg_reward, label=f"ε = {eps}")

plt.title("Average Reward over Steps for Different ε (ε-Greedy)")
plt.xlabel("Steps")
plt.ylabel("Average Reward")
plt.legend()
plt.grid(True)
plt.show()

# Single run plotting
_, avg_reward, opt = eGreedy(k=k, iterations=iterations, epsilon=0.2, seed=42)
steps = list(range(1, iterations + 1))

plt.figure(figsize=(10, 4))
plt.plot(steps, avg_reward, color='blue', label='Average Reward')
plt.xlabel('Steps')
plt.ylabel('Average Reward')
plt.title('Average Reward over Steps (Single Run)')
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(steps, opt, color='orange')
plt.xlabel('Steps')
plt.ylabel('% Optimal Action')
plt.title('Optimal Action Over Steps (Single Run)')
plt.grid(True)
plt.show()

# Average reward over multiple seeds
def average_reward_egreedy_over_seeds(epsilon, k=10, iterations=1000, reward_std=1.0, n_runs=100, seed_base=1000):
    all_final_rewards = []
    for i in range(n_runs):
        _, avg_reward, _ = eGreedy(k=k, iterations=iterations, epsilon=epsilon, reward_std=reward_std, seed=seed_base + i)
        all_final_rewards.append(avg_reward[-1])
    return np.mean(all_final_rewards)

# Run for multiple epsilon values
epsilon_values = [1/128, 1/64, 1/32, 1/16, 1/8, 1/4]
avg_rewards_1000_egreedy = [average_reward_egreedy_over_seeds(eps) for eps in epsilon_values]

print("Epsilon vs Avg Reward over 100 runs:", list(zip(epsilon_values, avg_rewards_1000_egreedy)))


## 2. Optimistic Initial Values
This method encourages exploration by setting initial action-value estimates $Q_1(a) = Q_0$ to a high (optimistic) value. Since these values are higher than the expected rewards, any selected action will likely disappoint the agent, driving down its estimate and encouraging the agent to try all other actions.

**Important Implementation Note**: A constant step-size parameter $\alpha$ (e.g., $\alpha = 0.1$) is required for optimistic initialization to have a lasting effect:
$$Q_{t+1}(A_t) = Q_t(A_t) + \alpha [R_t - Q_t(A_t)]$$
If a sample-average update ($1/N_t(A_t)$) is used, the initial value $Q_1(a)$ is completely overwritten by the first reward $R_1$, neutralizing the optimistic exploration effect.


In [ ]:
def greedy_optimistic(k=10, iterations=1000, Q0=5.0, alpha=0.1, reward_std=1.0, seed=None):
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    # True action values
    true_values = np.array([random.gauss(0, 1) for _ in range(k)])

    # Optimistic initial Q-values (constant step-size alpha ensures lasting effect)
    Q = np.full(k, Q0, dtype=float)
    N = np.zeros(k, dtype=int)  # counts

    average_reward = 0.0
    opt_count = 0
    best_action = np.argmax(true_values)

    rewards_list = []
    opt_fraction = []

    for t in range(1, iterations + 1):
        # Random tie-breaking
        max_Q = np.max(Q)
        action = np.random.choice(np.flatnonzero(Q == max_Q))

        # Generate reward
        reward = np.random.normal(true_values[action], reward_std)

        # Update estimates with constant step-size alpha
        N[action] += 1
        Q[action] += alpha * (reward - Q[action])

        # Update running average reward
        average_reward += (reward - average_reward) / t

        # Track optimal action fraction
        if action == best_action:
            opt_count += 1

        rewards_list.append(average_reward)
        opt_fraction.append(opt_count / t)

    return rewards_list, opt_fraction

# Average reward over multiple runs
def average_reward_over_seeds(Q0=1.0, k=10, iterations=1000, alpha=0.1, reward_std=1.0, n_runs=100, seed_base=1000):
    final_avg_rewards = []
    for i in range(n_runs):
        rewards, _ = greedy_optimistic(k=k, iterations=iterations, Q0=Q0, alpha=alpha, reward_std=reward_std, seed=seed_base + i)
        final_avg_rewards.append(rewards[-1])
    return np.mean(final_avg_rewards)

# Example usage
Q0_values = [0.25, 0.5, 1, 2, 4]
avg_rewards_1000_optimistic = [average_reward_over_seeds(Q0=Q0, n_runs=500) for Q0 in Q0_values]

print("Q0 vs Avg Reward:", list(zip(Q0_values, avg_rewards_1000_optimistic)))


## 3. Upper Confidence Bound (UCB) Action Selection
UCB selects actions based on both their estimated value and their uncertainty. It measures the potential of each action by adding an uncertainty bound to the estimate $Q_t(a)$:
$$A_t \doteq \underset{a}{\operatorname{argmax}} \left[ Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} \right]$$
The parameter $c > 0$ controls the degree of exploration. The square-root term represents a measure of uncertainty or variance in the estimate of action $a$.


In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

# Parameters
k = 10
iterations = 1000
seed_base = 1000

def upper_confidence_bound(k=10, iterations=1000, c=1.0, seed=None):
    # Seed for reproducibility
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    # True action values for this run
    true_action_values = np.random.randn(k)

    # Reward function
    def get_reward(action):
        return np.random.randn() + true_action_values[action]

    # Initialize Q-values and counts
    Q = np.zeros(k)
    N = np.zeros(k, dtype=int)

    rewards = []
    average_reward = []
    opt_count = 0
    opt = []

    best_action = np.argmax(true_action_values)

    # Initialize each action once to avoid divide-by-zero
    for j in range(k):
        Q[j] = get_reward(j)
        N[j] = 1
        if j == best_action:
            opt_count += 1
        opt.append(opt_count / (j + 1))
        rewards.append(Q[j])
        average_reward.append(np.mean(rewards))

    # Main UCB loop
    for i in range(k, iterations):
        UCB_values = Q + c * np.sqrt(np.log(i + 1) / N)
        # Random tie-breaking
        max_UCB = np.max(UCB_values)
        action = np.random.choice(np.flatnonzero(UCB_values == max_UCB))
        
        reward = get_reward(action)
        rewards.append(reward)
        average_reward.append(np.mean(rewards))

        if action == best_action:
            opt_count += 1
        opt.append(opt_count / (i + 1))

        # Update estimates
        N[action] += 1
        Q[action] += (reward - Q[action]) / N[action]

    return rewards, average_reward, opt

# ------------------------------------------------------------
# Averaging over multiple seeds
# ------------------------------------------------------------
def average_ucb_over_seeds(k=10, iterations=1000, c=1.0, n_seeds=150, seed_base=1000):
    # Compute average reward curve of UCB over multiple seeds.
    all_avg_rewards = []

    for i in range(n_seeds):
        seed = seed_base + i
        _, avg_reward, _ = upper_confidence_bound(k=k, iterations=iterations, c=c, seed=seed)
        all_avg_rewards.append(avg_reward)

    # Convert list of lists to array (shape: [n_seeds, iterations])
    all_avg_rewards = np.array(all_avg_rewards)
    
    # Average across seeds for each time step
    mean_avg_reward = np.mean(all_avg_rewards, axis=0)
    std_avg_reward = np.std(all_avg_rewards, axis=0)

    return mean_avg_reward, std_avg_reward

# ------------------------------------------------------------
# Run experiment for multiple c values and plot
# ------------------------------------------------------------
iterations = 1000
c_values_plot = [0.1, 0.5, 1.0, 2.0, 5.0]
k = 10
n_seeds = 150

plt.figure(figsize=(10, 6))

for c in c_values_plot:
    mean_reward, std_reward = average_ucb_over_seeds(k=k, iterations=iterations, c=c, n_seeds=n_seeds)
    steps = np.arange(1, iterations + 1)
    plt.plot(steps, mean_reward, label=f"c = {c}")
    plt.fill_between(steps, mean_reward - std_reward, mean_reward + std_reward, alpha=0.1)

plt.title("Average Reward over Steps (UCB) — Averaged over 150 Seeds")
plt.xlabel("Steps")
plt.ylabel("Average Reward")
plt.legend()
plt.grid(True)
plt.show()

# Average reward over multiple seeds for parameter study
def average_reward_ucb_over_seeds(c, k=10, iterations=1000, n_runs=100, seed_base=1000):
    final_rewards = []
    for i in range(n_runs):
        _, avg_reward, _ = upper_confidence_bound(k=k, iterations=iterations, c=c, seed=seed_base + i)
        final_rewards.append(avg_reward[-1])
    return np.mean(final_rewards)

# Parameter study
c_values = [1/16, 1/8, 1/4, 1/2, 1, 2, 4]
avg_reward_1000_ucb = [average_reward_ucb_over_seeds(c, n_runs=100) for c in c_values]

print("c values vs Avg Reward over 100 runs:", list(zip(c_values, avg_reward_1000_ucb)))


## 4. Gradient Bandit Algorithm
Instead of estimating action values, the gradient bandit algorithm learns a numerical preference $H_t(a)$ for each action. The action probabilities are determined using a softmax distribution:
$$\pi_t(a) \doteq \frac{e^{H_t(a)}}{\sum_{b=1}^k e^{H_t(b)}}$$
The preferences are updated using stochastic gradient ascent:
$$H_{t+1}(A_t) = H_t(A_t) + \alpha(R_t - \bar{R}_t)(1 - \pi_t(A_t))$$
$$H_{t+1}(a) = H_t(a) - \alpha(R_t - \bar{R}_t)\pi_t(a) \quad \text{for all } a \neq A_t$$
where $\bar{R}_t$ is the average of all rewards up to (but not including) time step $t$.


In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

# Parameters
k = 10
iterations = 1000
seed_base = 1000

def gradient_bandit(alpha, k=10, iterations=1000, seed=None):
    # Seed for reproducibility
    if seed is not None:
        np.random.seed(seed)
        random.seed(seed)

    # True action values for this run
    true_action_values = np.random.randn(k)

    # Reward function
    def get_reward(action):
        return np.random.randn() + true_action_values[action]

    # Action preferences
    H = np.zeros(k)
    rewards = []
    average_reward = []
    opt_count = 0
    opt = []

    best_action = np.argmax(true_action_values)

    for i in range(iterations):
        # Softmax probabilities
        exp_H = np.exp(H - np.max(H))
        prob = exp_H / np.sum(exp_H)

        # Select action according to probabilities
        action = np.random.choice(np.arange(k), p=prob)
        
        # Correct baseline calculation: average reward up to step i-1 (i.e. length i)
        if i == 0:
            baseline = 0.0
        else:
            baseline = np.mean(rewards)
            
        reward = get_reward(action)
        rewards.append(reward)

        # Track optimal action
        if action == best_action:
            opt_count += 1
        opt.append(opt_count / (i + 1))

        # Vectorized updates for preferences H
        H -= alpha * (reward - baseline) * prob
        H[action] += alpha * (reward - baseline)

        average_reward.append(np.mean(rewards))

    return rewards, average_reward, opt

# Average reward over multiple seeds
def average_reward_gb_over_seeds(alpha, k=10, iterations=1000, n_runs=100, seed_base=1000):
    final_rewards = []
    for i in range(n_runs):
        _, avg_reward, _ = gradient_bandit(alpha, k=k, iterations=iterations, seed=seed_base + i)
        final_rewards.append(avg_reward[-1])  # last step average reward
    return np.mean(final_rewards)

# Parameter study
alpha_values = [1/32, 1/16, 1/8, 1/4, 1/2, 1, 2, 4]
avg_reward_1000_gb = [average_reward_gb_over_seeds(alpha, n_runs=100) for alpha in alpha_values]

print("Alpha values vs Avg Reward over 100 runs:", list(zip(alpha_values, avg_reward_1000_gb)))


## 5. Parameter Study (Comparison)
We run a parameter study evaluating the performance of each algorithm over its first 1000 steps, varying their core parameter by factors of two. This reproduces Figure 2.6 from the textbook.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

# Plot each algorithm
plt.plot(epsilon_values, avg_rewards_1000_egreedy, marker='o', label='ε-greedy')
plt.plot(Q0_values, avg_rewards_1000_optimistic, marker='s', label='Greedy (optimistic Q₀)')
plt.plot(c_values, avg_reward_1000_ucb, marker='^', label='UCB')
plt.plot(alpha_values, avg_reward_1000_gb, marker='d', label='Gradient Bandit')

# Log scale on x-axis (commonly used in RL parameter studies)
plt.xscale('log')

# Labels and title
plt.xlabel('Parameter value (ε, Q₀, c, α)', fontsize=12)
plt.ylabel('Average reward at 1000 steps', fontsize=12)
plt.title('Parameter Study (Reproduction of Sutton & Barto Figure 2.6)', fontsize=14)

# Grid and legend
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()

plt.show()
